# Dafne Thigh Segmentation — Augmented Dataset (Lambda)

Runs the **Dafne Thigh model** 2D slice-by-slice on each of the 20 augmented NIfTI water volumes.

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
Output: `~/dafne_augmented_segs/{stem}/`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/path/to/dafne_model/" \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_model/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_augmented_segs/ \
  /path/to/local/dafne/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import os, subprocess, sys

# Reduce CUDA allocator fragmentation for the torch backend (only relevant if
# the loaded model turns out to be a DynamicTorchModel) — must be set before
# any CUDA context is created.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'dafne-dl', 'SimpleITK'])

# dafne-dl's own dependency list just says 'tensorflow' / 'torch' with no
# version pin, which on a fresh box does NOT guarantee GPU support:
#  - tensorflow>=2.13 needs the 'and-cuda' extra to pull in matching
#    CUDA/cuDNN pip packages (the old tensorflow-gpu package is gone).
#  - torch's default PyPI linux wheel does bundle CUDA, but pin it
#    explicitly so it can't silently get swapped for a CPU-only build.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'tensorflow[and-cuda]'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'torch', '--index-url', 'https://download.pytorch.org/whl/cu124'])

# tensorflow[and-cuda] pulls in numpy>=2, but this box's system 'ml_dtypes'
# package (/usr/lib/python3/dist-packages, a TF/JAX dependency baked into the
# image) is compiled against numpy 1.x and doesn't get reinstalled — importing
# tensorflow then crashes with "ImportError: numpy.core.umath failed to
# import". Pin numpy back down LAST so nothing after this re-upgrades it.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2'])

print('Dependencies installed.')

In [ ]:
import tensorflow as tf
import torch

# Must run before dafne_dl/any model touches a GPU — set_memory_growth raises
# if called after the device is already initialized.
print('--- TensorFlow ---')
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus)
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
if not gpus:
    print('  WARNING: TensorFlow sees no GPU — if the Thigh model is Keras-based '
          '(the default dafne_dl.DynamicDLModel), it will run on CPU.')

print('--- PyTorch ---')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(' ', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f'  VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
import glob, os
import numpy as np
import SimpleITK as sitk
from dafne_dl import DynamicDLModel

DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
OUTPUT_DIR = os.path.expanduser('~/dafne_augmented_segs')

_model_candidates = sorted(glob.glob(os.path.expanduser('~/dafne_model/*.model')))
if not _model_candidates:
    raise FileNotFoundError('No .model file found in ~/dafne_model/ — upload it first')
MODEL_PATH = _model_candidates[0]

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model : {MODEL_PATH}')
print(f'Found : {len(nii_files)} NIfTI water volumes')

In [ ]:
model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Model class:', type(model).__name__)
if hasattr(model, 'device'):
    print('  Torch device:', model.device)
print('Model loaded.')

In [ ]:
# Threshold below which a slice is considered empty (no tissue signal).
EMPTY_THRESHOLD = 0.01

for nii_path in nii_files:
    basename = os.path.basename(nii_path)
    # stem = filename without _water.nii.gz
    stem = basename.replace('_water.nii.gz', '')

    out_subdir = os.path.join(OUTPUT_DIR, stem)
    out_path   = os.path.join(out_subdir, f'{stem}_dafne_thigh.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    os.makedirs(out_subdir, exist_ok=True)

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(np.float32)  # (D, H, W)
    spacing   = img_sitk.GetSpacing()  # (sx, sy, sz) in SimpleITK order
    resolution = [spacing[1], spacing[0]]
    D, H, W = img_array.shape
    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')

    # Normalise to [0, 1] globally so Dafne's internal bias correction
    # sees a consistent intensity range and OtsuThreshold never divides by zero.
    v_min, v_max = float(img_array.min()), float(img_array.max())
    img_norm = (img_array - v_min) / (v_max - v_min + 1e-8)

    all_masks = {}
    skipped = 0
    for sl in range(D):
        slice_img = img_norm[sl]

        # Skip slices with no meaningful signal — these cause OtsuThreshold to fail.
        if float(slice_img.max()) < EMPTY_THRESHOLD:
            skipped += 1
            continue

        try:
            out = model({
                'image':            slice_img,
                'resolution':       resolution,
                'split_laterality': True,
                'classification':   'Thigh',
            })
        except RuntimeError as e:
            print(f'  [skip slice {sl}] {e}')
            skipped += 1
            continue

        for name, mask in out.items():
            if name not in all_masks:
                all_masks[name] = np.zeros((D, H, W), dtype=np.uint8)
            all_masks[name][sl] = np.asarray(mask, dtype=np.uint8)
        if (sl + 1) % 50 == 0 or sl == D - 1:
            print(f'  slice {sl+1}/{D}  (skipped so far: {skipped})')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}  muscles: {list(all_masks.keys())}  skipped: {skipped}/{D}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', '*.npz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    s = np.load(results[0])
    print(f'Sample: {results[0]}')
    for k in sorted(s.files):
        print(f'  {k}: {s[k].shape}  voxels={int(s[k].sum()):,}')